<a href="https://colab.research.google.com/github/HLZHarry/LLM-Practice/blob/main/ch01/ch01_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

**Does `transformers` connect to Hugging Face?**  
Yes — `transformers` is made by Hugging Face, and `from_pretrained()` downloads models directly from huggingface.co by default.

**Parameters:**
- `"microsoft/Phi-3-mini-4k-instruct"` — model ID on Hugging Face Hub (`owner/model-name`)
- `device_map="cuda"` — loads the model onto your GPU. Can also be `"cpu"` or `"auto"`
- `torch_dtype="auto"` — automatically picks the best precision (e.g. float16 on GPU, float32 on CPU) to save memory
- `trust_remote_code=True` — allows running custom Python code bundled with the model repo (required for non-standard architectures like Phi-3)

In [ ]:
from transformers import pipeline

# Create a pipeline
generator = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    return_full_text = False,
    max_new_tokens = 500,
    do_sample = False
)

**`pipeline()`** is a high-level wrapper that bundles model + tokenizer into a simple callable.

**Parameters:**
- `"text-generation"` — the task type; tells the pipeline how to process inputs/outputs
- `model=model` — passes the already-loaded model (reuses it instead of downloading again)
- `tokenizer=tokenizer` — passes the already-loaded tokenizer
- `return_full_text=False` — returns only the **newly generated** text, not the input prompt repeated back
- `max_new_tokens=500` — limits the response to 500 new tokens maximum
- `do_sample=False` — uses **greedy decoding** (always picks the highest probability next token), making output deterministic. Set to `True` for more varied/creative output

**Usage:** call `generator("your prompt here")` and it handles tokenization → inference → decoding automatically.

In [8]:
# The prompt (user input / query)
messages = [
    {"role": "user", "content": "Create a funny joke about chickens"}
]

# Generate output
output = generator(messages)
print(output)
print("----------------")
print(output[0]["generated_text"])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': ' Why did the chicken join the band? Because it had the drumsticks!'}]
----------------
 Why did the chicken join the band? Because it had the drumsticks!


## Why a Dictionary with `role` and `content`?
```python
messages = [
    {"role": "user", "content": "Create a funny joke about chickens"}
]
```

**This format is called the "Chat Template" or "ChatML" format.**

- It's not just for Phi-3 — it's the **standard format across most modern chat/instruction models** (GPT, LLaMA, Mistral, Gemini, etc.)
- The `role` field tells the model **who is speaking**:
  - `"user"` — the human input
  - `"assistant"` — the model's response
  - `"system"` — optional instructions/persona for the model
- The `content` field holds the **actual message text**
- Using a list allows **multi-turn conversations** by adding more dicts, e.g.:
```python
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hello!"},
    {"role": "assistant", "content": "Hi! How can I help?"},
    {"role": "user", "content": "Tell me a joke"}
]
```

The model uses this history to maintain conversation context.

---

## Why is the Output a List of Dictionaries?
```python
output = generator(messages)
print(output[0]["generated_text"])
```

- The pipeline **always returns a list** because it's designed to handle **batched inputs** (multiple prompts at once). Even for a single input, it wraps the result in a list for consistency.
- Each list item is a **dictionary** containing the result fields — in this case `"generated_text"`
- So `output[0]` gets the first (and only) result, and `["generated_text"]` extracts the text

**Example of what `output` looks like:**
```python
[
    {"generated_text": "Why did the chicken cross the road? ..."}
]
```

If you passed in a batch of 3 prompts, `output` would have 3 dictionaries.